In [ ]:
# Core scverse libraries
import os
import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt





# === INPUT PATHS ===
msi_input_folder = "/home/ajarrah/PhD_Thesis/chapter_2/h5ad_data_processed_4lockmasses_filtered_halfbrain/"
msi_common_mzs_csv_input_path = "/home/ajarrah/PhD_Thesis/chapter_2/csv_data/common_mz_clusters_improved.csv"

rna_input_file = '/home/ajarrah/PhD_Thesis/chapter_3/aggregated_h5ad_data/aggregated_mouse_brain_202502.h5ad'
rna_input_folder = "/home/ajarrah/PhD_Thesis/chapter_3/h5ad_data/"
rna_tissue_positions_csv_input_path = "/home/ajarrah/PhD_Thesis/chapter_4/tissue_positions/tissue_positions.csv"

lipid_image_output_dir = "/home/ajarrah/PhD_Thesis/chapter_4/images/lipids/"
gene_image_output_dir = "/home/ajarrah/PhD_Thesis/chapter_4/images/genes/"

rna_spot_spacing = 100  # µm center-to-center
msi_pixel_spacing = 60  # µm center-to-center

In [ ]:
# Your list of sample IDs (also used as keys in the dictionary)
msi_aad_1 = sc.read_h5ad(os.path.join(msi_input_folder, "halfbrain_aad_1_filtered.h5ad"))
msi_aad_2 = sc.read_h5ad(os.path.join(msi_input_folder, "halfbrain_aad_2_filtered.h5ad"))
msi_aad_3 = sc.read_h5ad(os.path.join(msi_input_folder, "halfbrain_aad_3_filtered.h5ad"))
msi_aad_4 = sc.read_h5ad(os.path.join(msi_input_folder, "halfbrain_aad_4_filtered.h5ad"))
msi_ac_1 = sc.read_h5ad(os.path.join(msi_input_folder, "halfbrain_ac_1_filtered.h5ad"))
msi_ac_2 = sc.read_h5ad(os.path.join(msi_input_folder, "halfbrain_ac_2_filtered.h5ad"))
msi_ac_3 = sc.read_h5ad(os.path.join(msi_input_folder, "halfbrain_ac_3_filtered.h5ad"))
msi_ac_4 = sc.read_h5ad(os.path.join(msi_input_folder, "halfbrain_ac_4_filtered.h5ad"))
msi_yad_1 = sc.read_h5ad(os.path.join(msi_input_folder, "halfbrain_yad_1_filtered.h5ad"))
msi_yad_2 = sc.read_h5ad(os.path.join(msi_input_folder, "halfbrain_yad_2_filtered.h5ad"))
msi_yad_3 = sc.read_h5ad(os.path.join(msi_input_folder, "halfbrain_yad_3_filtered.h5ad"))
msi_yad_4 = sc.read_h5ad(os.path.join(msi_input_folder, "halfbrain_yad_4_filtered.h5ad"))
msi_yc_1 = sc.read_h5ad(os.path.join(msi_input_folder, "halfbrain_yc_1_filtered.h5ad"))
msi_yc_2 = sc.read_h5ad(os.path.join(msi_input_folder, "halfbrain_yc_2_filtered.h5ad"))
msi_yc_3 = sc.read_h5ad(os.path.join(msi_input_folder, "halfbrain_yc_3_filtered.h5ad"))
msi_yc_4 = sc.read_h5ad(os.path.join(msi_input_folder, "halfbrain_yc_4_filtered.h5ad"))

msi_sample_list = [
    msi_aad_1, msi_aad_2, msi_aad_3, msi_aad_4,
    msi_ac_1, msi_ac_2, msi_ac_3, msi_ac_4,
    msi_yad_1, msi_yad_2, msi_yad_3, msi_yad_4,
    msi_yc_1, msi_yc_2, msi_yc_3, msi_yc_4
]

msi_sample_ids = [
    "aad_1", "aad_2", "aad_3", "aad_4",
    "ac_1", "ac_2", "ac_3", "ac_4",
    "yad_1", "yad_2", "yad_3", "yad_4",
    "yc_1", "yc_2", "yc_3", "yc_4"
]

# Map sample ID → AnnData object
msi_sample_map = dict(zip(msi_sample_ids, msi_sample_list))

# Assuming sample_list contains your AnnData objects and their variable names are like 'aad_1', 'ac_1', etc.
# They must be defined variables or in a dictionary for easy iteration.

# Example: if you have them in a dictionary like sample_map = {'aad_1': aad_1, ...}

for sample_id in msi_sample_list:
    msi_adata = sample_id  # Get the AnnData object by variable name string
    # Sum all intensities for each spot (row) across all m/z (columns)
    tic = msi_adata.X.sum(axis=1)  # shape (n_spots, 1) or (n_spots,) depending on sparse or dense
    
    # If sparse, convert to flat array
    if hasattr(tic, "A1"):  # sparse matrix
        tic = tic.A1
    else:
        tic = np.asarray(tic).flatten()

    # Add TIC as a new column in obs
    msi_adata.obs["Processed_TIC"] = tic
    

# === STEP 3: Load common m/z list ===
common_mz_df = pd.read_csv(msi_common_mzs_csv_input_path)


# Example: convert group/sample to sample_id that matches your sample_map keys
def make_sample_id(row):
    # lower group, then underscore, then sample number
    return f"{row['group'].lower()}_{row['sample']}"

common_mz_df['sample_id'] = common_mz_df.apply(make_sample_id, axis=1)

pivot_df_msi = common_mz_df.pivot(index='sample_id', columns='common_group_name', values='mzs')

msi_order = [
    'yc_1', 'yc_2', 'yc_3', 'yc_4',
    'yad_1', 'yad_2', 'yad_3', 'yad_4',
    'ac_1', 'ac_2', 'ac_3', 'ac_4',
    'aad_1', 'aad_2', 'aad_3', 'aad_4'
]

# Assuming pivot_df is your pivoted DataFrame with index = sample_id
pivot_df_msi_sorted = pivot_df_msi.loc[pivot_df_msi.index.intersection(msi_order)]
pivot_df_msi_sorted = pivot_df_msi_sorted.loc[msi_order]

pivot_df_msi_sorted
pivot_df_msi_sorted.columns


for msi_sample in msi_sample_list:
    # Extract spatial coordinates
    spatial = msi_sample.obsm['spatial']

    array_row = spatial[:, 1]  # or [:, 0] depending on order (check which is row vs col)
    array_col = spatial[:, 0]

    x_um = array_col * (msi_pixel_spacing)
    y_um = array_row * (msi_pixel_spacing)

    msi_sample.obs['x_um'] = x_um
    msi_sample.obs['y_um'] = y_um

In [ ]:
rna_adata = sc.read_h5ad(rna_input_file)

rna_aad_1 = sc.read_h5ad(os.path.join(rna_input_folder, "A1_Aged_AD_Mouse_Brain_202502.h5ad"))
rna_aad_2 = sc.read_h5ad(os.path.join(rna_input_folder, "B1_Aged_AD_Mouse_Brain_202502.h5ad"))
rna_aad_3 = sc.read_h5ad(os.path.join(rna_input_folder, "C1_Aged_AD_Mouse_Brain_202502.h5ad"))
rna_aad_4 = sc.read_h5ad(os.path.join(rna_input_folder, "D1_Aged_AD_Mouse_Brain_202502.h5ad"))
rna_ac_1 = sc.read_h5ad(os.path.join(rna_input_folder, "A1_Aged_Control_Mouse_Brain_202502.h5ad"))
rna_ac_2 = sc.read_h5ad(os.path.join(rna_input_folder, "B1_Aged_Control_Mouse_Brain_202502.h5ad"))
rna_ac_3 = sc.read_h5ad(os.path.join(rna_input_folder, "C1_Aged_Control_Mouse_Brain_202502.h5ad"))
rna_ac_4 = sc.read_h5ad(os.path.join(rna_input_folder, "D1_Aged_Control_Mouse_Brain_202502.h5ad"))
rna_yad_1 = sc.read_h5ad(os.path.join(rna_input_folder, "A1_Young_AD_Mouse_Brain_202502.h5ad"))
rna_yad_2 = sc.read_h5ad(os.path.join(rna_input_folder, "B1_Young_AD_Mouse_Brain_202502.h5ad"))
rna_yad_3 = sc.read_h5ad(os.path.join(rna_input_folder, "C1_Young_AD_Mouse_Brain_202502.h5ad"))
rna_yad_4 = sc.read_h5ad(os.path.join(rna_input_folder, "D1_Young_AD_Mouse_Brain_202502.h5ad"))
rna_yc_1 = sc.read_h5ad(os.path.join(rna_input_folder, "A1_Young_Control_Mouse_Brain_202502.h5ad"))
rna_yc_2 = sc.read_h5ad(os.path.join(rna_input_folder, "B1_Young_Control_Mouse_Brain_202502.h5ad"))
rna_yc_3 = sc.read_h5ad(os.path.join(rna_input_folder, "C1_Young_Control_Mouse_Brain_202502.h5ad"))
rna_yc_4 = sc.read_h5ad(os.path.join(rna_input_folder, "D1_Young_Control_Mouse_Brain_202502.h5ad"))

rna_data_name = [rna_yc_1, rna_yc_2, rna_yc_3, rna_yc_4,
                rna_yad_1, rna_yad_2, rna_yad_3, rna_yad_4,
                rna_ac_1, rna_ac_2, rna_ac_3, rna_ac_4,
                rna_aad_1, rna_aad_2, rna_aad_3, rna_aad_4]

rna_sample_name = ["YC_1", "YC_2", "YC_3", "YC_4",
                "YAD_1", "YAD_2", "YAD_3", "YAD_4",
                "AC_1", "AC_2", "AC_3", "AC_4",
                "AAD_1", "AAD_2", "AAD_3", "AAD_4"]

rna_sample_display_name = ["Young Control 1", "Young Control 2", "Young Control 3", "Young Control 4",
                       "Young AD 1", "Young AD 2", "Young AD 3", "Young AD 4",
                       "Aged Control 1", "Aged Control 2", "Aged Control 3", "Aged Control 4",
                       "Aged AD 1", "Aged AD 2", "Aged AD 3", "Aged AD 4"]

# Ensure matrix is dense or use .A1 for sparse matrices
if isinstance(rna_adata.X, np.ndarray):
    non_zero_genes = rna_adata.X.sum(axis=0) != 0
else:
    non_zero_genes = np.array((rna_adata.X.sum(axis=0) != 0)).ravel()  # for sparse matrix

rna_adata = rna_adata[:, non_zero_genes].copy()


# Normalize total counts per cell (CPM / library size normalization)
sc.pp.normalize_total(rna_adata, target_sum=1e2)   # or 1e6 depending on preference
for data in rna_data_name:
    sc.pp.normalize_total(data, target_sum=1e2)   # or 1e6 depending on preference

# Read the CSV
# Read only selected columns
tissue_positions = pd.read_csv(
    rna_tissue_positions_csv_input_path,
    usecols=["barcode", "array_row", "array_col"])

# Show first rows
print(tissue_positions.head())


rna_spot_spacing = 100  # µm center-to-center
n_rows = tissue_positions['array_row'].max() + 1
n_cols = tissue_positions['array_col'].max() + 1

coords_um = []
for _, row in tissue_positions.iterrows():
    r, c = row['array_row'], row['array_col']
    y = r * (np.sqrt(3)/2 * rna_spot_spacing)
    x = c * rna_spot_spacing/2 
    coords_um.append((x, y))

coords_um = np.array(coords_um)
tissue_positions['x_um'] = coords_um[:, 0]
tissue_positions['y_um'] = coords_um[:, 1]

In [ ]:
import numpy as np
import scanpy as sc
import os

# --- Step 1: count how many samples each gene is lowly expressed in ---
all_genes = rna_data_name[0].var_names
gene_low_counts = {gene: 0 for gene in all_genes}

for adata in rna_data_name:
    spot_counts = np.array((adata.X > 0).sum(axis=0)).ravel()
    low_expr_mask = spot_counts <= 500
    low_genes = adata.var_names[low_expr_mask]
    for gene in low_genes:
        gene_low_counts[gene] += 1

# --- Step 2: identify genes lowly expressed (≤10 spots) in ≥15 samples ---
low_expr_genes_across_15 = [gene for gene, count in gene_low_counts.items() if count >= 12]

print(f"Number of genes expressed in ≤10 spots in ≥15 samples: {len(low_expr_genes_across_15)}")

# --- Step 3: filter them out ---
for i, adata in enumerate(rna_data_name):
    genes_to_keep = ~adata.var_names.isin(low_expr_genes_across_15)
    filtered = adata[:, genes_to_keep].copy()
    rna_data_name[i] = filtered
    print(f"{rna_sample_name[i]}: filtered -> {filtered.n_vars} genes remain")

# --- Step 4 (optional): save filtered files ---
# for adata, name in zip(rna_data_name, rna_sample_name):
#     out_path = os.path.join(rna_input_folder, f"{name}_filtered.h5ad")
#     adata.write(out_path)
#     print(f"Saved: {out_path}")


In [ ]:
from matplotlib.colors import LinearSegmentedColormap

# === COLOR SCALE ===
custom_cmap = LinearSegmentedColormap.from_list("custom_scale", [
    (0.0, "#454545"),      # dark gray
    (0.00000001, "#000000"),  # black
    (0.10, "#000080"),     # navy
    (0.15, "#0000FF"),     # blue
    (0.30, "#8000FF"),     # purple-ish
    (0.45, "#FF0000"),     # red
    (0.60, "#FF8000"),     # orange
    (0.75, "#FFFF00"),     # yellow
    (1.0, "#FFFFFF")       # white
])


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt


def msi_plot_producer(
    adata,
    mz_val,
    sample_name="YC_1",
    output_dir="plots",
    tol=0.001,
    vmax_percentile=99.9,
    square_size=60,   # µm per pixel (both x and y)
    cmap="viridis",
    show_colorbar=True,
    show_axis=True,
    show_title=True,
    background_color="#00BF00"
):
    """
    Plot TIC-normalized intensity for one m/z and one sample, with each square
    representing 'square_size' µm × 'square_size' µm.
    The figure is saved with a structured filename.
    """

    # --- Extract m/z index ---
    mz_axis = adata.var_names.astype(float).values
    mz_diff = np.abs(mz_axis - mz_val)
    if np.min(mz_diff) > tol:
        print(f"m/z {mz_val} not found within tolerance {tol}")
        return
    mz_index = np.argmin(mz_diff)

    # --- Extract intensities ---
    intensities = (
        adata.X[:, mz_index].toarray().flatten()
        if hasattr(adata.X, "toarray")
        else adata.X[:, mz_index]
    )

    # --- Extract coordinates & TIC ---
    x = adata.obs["x_um"].values
    y = adata.obs["y_um"].values
    tic = adata.obs["Processed_TIC"].values

    # Normalize by TIC
    normalized_intensities = (intensities / tic) * 100
    normalized_intensities = np.nan_to_num(normalized_intensities, nan=0.0, posinf=0.0, neginf=0.0)

    # --- Build a grid ---
    x_unique = np.unique(x)
    y_unique = np.unique(y)
    x_size = len(x_unique)
    y_size = len(y_unique)

    grid = np.full((y_size, x_size), np.nan)
    for xi, yi, val in zip(x, y, normalized_intensities):
        x_idx = np.where(x_unique == xi)[0][0]
        y_idx = np.where(y_unique == yi)[0][0]
        grid[y_idx, x_idx] = val

    # --- Color scaling ---
    vmin_value = np.nanmin(normalized_intensities)
    vmax_value = np.percentile(normalized_intensities, vmax_percentile)

    # --- Plot ---
    extent = [x.min(), x.max() + square_size, y.min(), y.max() + square_size]
    fig, ax = plt.subplots(
        figsize=(x_size * square_size / 1000, y_size * square_size / 1000)
    )
    fig.patch.set_facecolor(background_color)  # figure background (light green)

    im = ax.imshow(
        grid,
        cmap=cmap,
        origin="lower",
        extent=extent,
        vmin=vmin_value,
        vmax=vmax_value,
        interpolation="none",
    )

    ax.invert_xaxis()  # 🔄 Reverse x-axis direction
    ax.set_aspect("equal")  # Equal aspect ratio

    # Optional components
    if show_axis:
        ax.set_xlabel("x (µm)")
        ax.set_ylabel("y (µm)")
    else:
        ax.axis("off")

    if show_title:
        ax.set_title(f"m/z {mz_val:.4f} normalized intensity")

    if show_colorbar:
        cbar = fig.colorbar(im, ax=ax)
        cbar.set_label(f"Normalized intensity (TIC%)")

    # --- Save image ---
    sample_dir = os.path.join(output_dir, sample_name)
    os.makedirs(sample_dir, exist_ok=True)
    filename = f"{mz_val:.4f}|{sample_name}|{vmin_value:.2f}|{vmax_value:.2f}.png"
    filepath = os.path.join(sample_dir, filename)
    fig.savefig(filepath, dpi=300, bbox_inches="tight", transparent=False)
    plt.close(fig)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.patches import Circle, Polygon
from scipy.spatial import cKDTree

def rna_plot_producer_interpolater(
    adata,
    tissue_positions,
    gene,
    rna_sample_name="YC_1",
    output_dir="plots",
    spot_diameter=55,
    cmap_name="viridis",
    figsize=(12, 12),
    alpha_rhombi=0.7,
    draw_poly_edges=False,
    draw_spot_edges=True,
    show_colorbar=True,
    show_axis=True,
    show_title=True,
    background_color="#00BF00"
):
    """
    Plot spatial transcriptomics spots with interpolated rhombus regions
    and save the figure as an image.

    Parameters
    ----------
    adata : AnnData
        Expression data (must include `var_names` for genes).
    tissue_positions : pd.DataFrame
        DataFrame with columns ['barcode', 'x_um', 'y_um'].
    gene : str
        Gene to visualize.
    rna_sample_name : str
        Name of the RNA sample for naming the file.
    output_dir : str
        Base directory to save images.
    show_colorbar : bool
        Whether to include the colorbar.
    show_axis : bool
        Whether to show axis lines, labels, and ticks.
    show_title : bool
        Whether to display a title.
    """

    # --- Step 1: extract data ---
    spot_radius = spot_diameter / 2
    x = tissue_positions["x_um"].values
    y = tissue_positions["y_um"].values

    expr = adata[:, adata.var_names == gene].X.toarray().flatten()
    barcode_to_expr = dict(zip(adata.obs_names, expr))
    expr_full = tissue_positions["barcode"].map(barcode_to_expr).to_numpy()

    vmin, vmax = np.nanmin(expr_full), np.nanmax(expr_full)

    # --- Step 2: build KDTree ---
    coords = np.column_stack([x, y])
    tree = cKDTree(coords)

    # --- Step 3: compute rhombi ---
    mid_polys, mid_vals = [], []
    for i, (xi, yi, vi) in enumerate(zip(x, y, expr_full)):
        if np.isnan(vi):
            continue
        dists, idxs = tree.query([xi, yi], k=7)  # self + 6 neighbors
        for j in idxs[1:]:
            if np.isnan(expr_full[j]):
                continue
            xj, yj, vj = x[j], y[j], expr_full[j]

            # Midpoint and averaged value
            mx, my = (xi + xj) / 2, (yi + yj) / 2
            mv = (vi + vj) / 2

            dx, dy = xj - xi, yj - yi
            length = np.sqrt(dx**2 + dy**2)
            if length == 0:
                continue

            px, py = -dy / length, dx / length
            half_short = length / (2 * np.sqrt(3))

            p1 = (mx - dx / 2, my - dy / 2)
            p2 = (mx + dx / 2, my + dy / 2)
            p3 = (mx + px * half_short, my + py * half_short)
            p4 = (mx - px * half_short, my - py * half_short)

            mid_polys.append([p1, p3, p2, p4])
            mid_vals.append(mv)

    # --- Step 4: plot ---
    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor(background_color)
    ax.set_aspect("equal")

    cmap = plt.get_cmap(cmap_name) if isinstance(cmap_name, str) else cmap_name
    norm = Normalize(vmin=vmin, vmax=vmax)

    # rhombi
    for poly, val in zip(mid_polys, mid_vals):
        patch = Polygon(
            poly,
            closed=True,
            facecolor=cmap(norm(val)),
            edgecolor="k" if draw_poly_edges else "none",
            lw=0.2,
            alpha=alpha_rhombi,
        )
        ax.add_patch(patch)

    # circles
    for xi, yi, val in zip(x, y, expr_full):
        if not np.isnan(val):
            circle = Circle(
                (xi, yi),
                radius=spot_radius,
                facecolor=cmap(norm(val)),
                edgecolor="k" if draw_spot_edges else "none",
                lw=0.2,
            )
            ax.add_patch(circle)

    ax.set_xlim(x.min() - 2 * spot_radius, x.max() + 2 * spot_radius)
    ax.set_ylim(y.min() - 2 * spot_radius, y.max() + 2 * spot_radius)
    ax.invert_yaxis()

    # Optional components
    if show_axis:
        ax.set_xlabel("x (µm)")
        ax.set_ylabel("y (µm)")
    else:
        ax.axis("off")

    if show_title:
        ax.set_title(f"{gene} expression (spots + interpolated rhombi)")

    if show_colorbar:
        sm = ScalarMappable(norm=norm, cmap=cmap)
        sm.set_array([])
        fig.colorbar(sm, ax=ax, label=f"{gene} expression")

    # --- Step 5: save figure ---
    sample_dir = os.path.join(output_dir, rna_sample_name)
    os.makedirs(sample_dir, exist_ok=True)
    filename = f'{gene}|{rna_sample_name}|{vmin:.2f}|{vmax:.2f}.png'
    fig.savefig(os.path.join(sample_dir, filename), dpi=300, bbox_inches='tight', transparent=False)
    plt.close(fig)

In [ ]:
import os
import numpy as np
import scanpy as sc
import psutil
from concurrent.futures import ProcessPoolExecutor, as_completed

# ============================
# User Settings (constants)
# ============================
RNA_INPUT_FOLDER = rna_input_folder
OUTPUT_DIR = gene_image_output_dir
MEMORY_PER_WORKER_GB = 2   # estimated memory used per worker
LOG_INTERVAL = 1000        # print progress every N genes

# ============================
# Helper function
# ============================
def process_gene(args):
    """Worker function: processes a single gene."""
    adata_path, tissue_positions, gene, sample_name, output_dir, cmap = args

    # Load adata in backed mode (read-only, memory efficient)
    adata = sc.read_h5ad(adata_path, backed='r')

    # Extract gene index
    if gene not in adata.var_names:
        return f"{sample_name}: gene {gene} not found"
    idx = np.where(adata.var_names == gene)[0][0]

    # Get expression vector (sparse or dense)
    expr = adata.X[:, idx].toarray().ravel() if hasattr(adata.X, "toarray") else adata.X[:, idx]

    # --- your gene-specific processing here ---
    result = f"{sample_name}:{gene}"
    return result

# ============================
# Main
# ============================
if __name__ == "__main__":
    # Detect system resources
    total_cores = os.cpu_count()
    total_memory_gb = psutil.virtual_memory().total / (1024**3)
    print(f"🧠 Detected system: {total_cores} CPU cores | {total_memory_gb:.1f} GB RAM\n")

    # Ask user for resource limits
    use_cores = int(input(f"Enter number of CPU cores to use (1–{total_cores}): ").strip() or total_cores - 1)
    use_memory_gb = float(input(f"Enter total memory to use in GB (≤ {total_memory_gb:.1f}): ").strip() or 16)

    # Determine safe number of parallel workers
    max_workers = min(use_cores, int(use_memory_gb / MEMORY_PER_WORKER_GB))
    print(f"\n⚙️ Using up to {max_workers} workers "
          f"(limit: {use_memory_gb} GB memory, {use_cores} CPU cores)\n")

    # Prepare task list
    tasks = []
    for adata, sample_name in zip(rna_data_name, rna_sample_name):
        print(f"🧬 Preparing RNA sample: {sample_name}")
        adata_path = os.path.join(RNA_INPUT_FOLDER, f"{sample_name}.h5ad")
        for gene in adata.var_names:
            tasks.append((adata_path, tissue_positions, gene, sample_name, OUTPUT_DIR, custom_cmap))

    total_tasks = len(tasks)
    print(f"\nTotal genes to process: {total_tasks}\n")

    # Start memory- and CPU-aware multiprocessing
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(process_gene, task) for task in tasks]

        for i, f in enumerate(as_completed(futures), 1):
            f.result()  # optional: handle or store result
            if i % LOG_INTERVAL == 0 or i == total_tasks:
                mem = psutil.virtual_memory()
                print(f"✅ {i}/{total_tasks} genes processed | "
                      f"Memory used: {mem.percent:.1f}% ({mem.used/1e9:.1f} GB)")

    print("\n🎉 All genes processed successfully within resource limits.")


In [ ]:
 #--- Loop through all samples ---
for adata, sample_id in zip(msi_sample_list, msi_sample_ids):
    print(f"\n🧠 Processing sample: {sample_id}")

    # Retrieve all aligned m/z values for this sample from the DataFrame
    if sample_id not in pivot_df_msi_sorted.index:
        print(f"⚠️ Sample '{sample_id}' not found in common_mzs_pivot_sorted — skipping.")
        continue

    mz_values = pivot_df_msi_sorted.loc[sample_id].values.astype(float)

    for mz_val in mz_values:
        try:
            msi_plot_producer(
                adata=adata,
                mz_val=mz_val,
                sample_name=sample_id,
                output_dir=lipid_image_output_dir,
                tol=0.01,
                vmax_percentile=99.8,
                square_size=60,  # µm per pixel
                cmap=custom_cmap,
                show_colorbar=False,
                show_axis=False,
                show_title=False,
                background_color="#00BF00"
            )
        except Exception as e:
            print(f"⚠️ Skipped m/z {mz_val:.4f} in {sample_id} due to error: {e}")

    print(f"✅ Finished sample: {sample_id}")

print("\n🎉 All aligned m/z values processed for all samples.")

In [ ]:
# --- Loop through all RNA samples ---
for adata, sample_name in zip(rna_data_name, rna_sample_name):
    print(f"\n🧬 Processing RNA sample: {sample_name}")

    # Extract all gene names from AnnData
    gene_list = adata.var_names.tolist()

    for gene in gene_list:
        try:
            rna_plot_producer_interpolater(
                adata=adata,
                tissue_positions=tissue_positions,
                gene=gene,
                rna_sample_name=sample_name,
                output_dir=gene_image_output_dir,
                spot_diameter=55,
                cmap_name=custom_cmap,
                figsize=(10, 10),
                alpha_rhombi=1,
                draw_poly_edges=False,
                draw_spot_edges=False,
                show_colorbar=False,
                show_axis=False,
                show_title=False,
                background_color="#00BF00"
            )
        except Exception as e:
            print(f"⚠️ Skipped gene {gene} in {sample_name} due to error: {e}")

    print(f"✅ Finished sample: {sample_name}")

print("\n🎉 All genes processed for all RNA samples.")

In [ ]:
import os
from concurrent.futures import ProcessPoolExecutor, as_completed

def process_gene(args):
    """Function to run in each process"""
    adata, tissue_positions, gene, sample_name, output_dir, custom_cmap = args
    try:
        rna_plot_producer_interpolater(
            adata=adata,
            tissue_positions=tissue_positions,
            gene=gene,
            rna_sample_name=sample_name,
            output_dir=output_dir,
            spot_diameter=55,
            cmap_name=custom_cmap,
            figsize=(10, 10),
            alpha_rhombi=1,
            draw_poly_edges=False,
            draw_spot_edges=False,
            show_colorbar=False,
            show_axis=False,
            show_title=False,
            background_color="#00BF00"
        )
        return f"✅ {sample_name} | {gene}"
    except Exception as e:
        return f"⚠️ Skipped {sample_name} | {gene}: {e}"

if __name__ == "__main__":
    max_workers = os.cpu_count() - 2  # leave 1 CPU free
    total_tasks = sum(len(adata.var_names) for adata in rna_data_name)
    completed = 0

    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futures = []
        for adata, sample_name in zip(rna_data_name, rna_sample_name):
            print(f"\n🧬 Processing RNA sample: {sample_name}")
            gene_list = adata.var_names.tolist()

            for gene in gene_list:
                futures.append(
                    executor.submit(
                        process_gene,
                        (adata, tissue_positions, gene, sample_name, gene_image_output_dir, custom_cmap)
                    )
                )

        for i, f in enumerate(as_completed(futures), 1):
            result = f.result()
            completed += 1

            # Print progress every 1000 completed tasks
            if completed % 1000 == 0 or completed == total_tasks:
                print(f"✅ {completed}/{total_tasks} genes processed...")

    print("\n🎉 All genes processed for all RNA samples.")

In [ ]:
def process_mz(args):
    """Function to run in each process"""
    adata, mz_val, sample_id, output_dir, custom_cmap = args
    try:
        msi_plot_producer(
            adata=adata,
            mz_val=mz_val,
            sample_name=sample_id,
            output_dir=output_dir,
            tol=0.01,
            vmax_percentile=99.8,
            square_size=60,
            cmap=custom_cmap,
            show_colorbar=False,
            show_axis=False,
            show_title=False,
            background_color="#00BF00"
        )
        return f"✅ {sample_id} | m/z {mz_val:.4f}"
    except Exception as e:
        return f"⚠️ Skipped {sample_id} | m/z {mz_val:.4f}: {e}"

if __name__ == "__main__":
    max_workers = os.cpu_count() - 2

    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futures = []
        for adata, sample_id in zip(msi_sample_list, msi_sample_ids):
            print(f"\n🧠 Processing sample: {sample_id}")

            if sample_id not in pivot_df_msi_sorted.index:
                print(f"⚠️ Sample '{sample_id}' not found — skipping.")
                continue

            mz_values = pivot_df_msi_sorted.loc[sample_id].values.astype(float)

            for mz_val in mz_values:
                futures.append(
                    executor.submit(
                        process_mz,
                        (adata, mz_val, sample_id, lipid_image_output_dir, custom_cmap)
                    )
                )

        for i, f in enumerate(as_completed(futures), 1):
            print(f"{i}/{len(futures)} — {f.result()}")

    print("\n🎉 All aligned m/z values processed for all samples.")
